In [1]:
import pandas as pd

df_original = pd.read_csv('/content/mobile phone price prediction.csv')

print("First 5 rows of the DataFrame:")
print(df_original.head())

print("\nConcise summary of the DataFrame:")
df_original.info()

First 5 rows of the DataFrame:
   index                                  Name  Rating  Spec_score  \
0      0                 Samsung Galaxy F14 5G    4.65          68   
1      1                    Samsung Galaxy A11    4.20          63   
2      2                    Samsung Galaxy A13    4.30          75   
3      3                    Samsung Galaxy F23    4.10          73   
4      4  Samsung Galaxy A03s (4GB RAM + 64GB)    4.10          69   

                       No_of_sim       Ram            Battery     Display  \
0  Dual Sim, 3G, 4G, 5G, VoLTE,   4 GB RAM  6000 mAh Battery   6.6 inches   
1      Dual Sim, 3G, 4G, VoLTE,   2 GB RAM  4000 mAh Battery   6.4 inches   
2      Dual Sim, 3G, 4G, VoLTE,   4 GB RAM  5000 mAh Battery   6.6 inches   
3      Dual Sim, 3G, 4G, VoLTE,   4 GB RAM   6000 mAh Battery  6.4 inches   
4      Dual Sim, 3G, 4G, VoLTE,   4 GB RAM  5000 mAh Battery   6.5 inches   

                                              Camera  \
0    50 MP + 2 MP Dual Rear &

In [2]:
categorical_cols = df_original.select_dtypes(include='object').columns
numerical_cols = df_original.select_dtypes(include=['int64', 'float64']).columns

print("\nUnique values and counts for categorical columns:")
for col in categorical_cols:
    print(f"\nColumn: {col}")
    print(df_original[col].value_counts(dropna=False))

print("\nDescriptive statistics for numerical columns:")
print(df_original[numerical_cols].describe())


Unique values and counts for categorical columns:

Column: Name
Name
Realme Narzo N55 (6GB RAM + 128GB)    3
Vivo S19 Pro                          2
Samsung Galaxy S21 FE (Snapdragon)    2
Vivo V25 Pro (12GB RAM + 256GB)       2
Vivo V30 5G                           2
                                     ..
Poco C55                              1
Poco C61 (6GB RAM + 128GB)            1
iQOO 9 Pro 5G (12GB RAM + 256GB)      1
iQOO 13 Pro                           1
Poco C65 (8GB RAM + 256GB)            1
Name: count, Length: 1334, dtype: int64

Column: No_of_sim
No_of_sim
Dual Sim, 3G, 4G, 5G, VoLTE,             818
Dual Sim, 3G, 4G, VoLTE,                 419
Dual Sim, 3G, 4G, 5G, VoLTE, Vo5G,        91
Single Sim, 3G, 4G, 5G, VoLTE,            20
Dual Sim, 3G, 4G,                         13
Single Sim, 3G, 4G, VoLTE,                 6
No Sim Supported,                          1
Single Sim, 3G, 4G, 5G, VoLTE, Vo5G,       1
Dual Sim, 3G, VoLTE,                       1
Name: count, dty

In [3]:
df_cleaned = df_original.copy()

# 1. Convert 'Price' column to numerical format
df_cleaned['Price'] = df_cleaned['Price'].str.replace(',', '', regex=False)
df_cleaned['Price'] = pd.to_numeric(df_cleaned['Price'], errors='coerce')

# Fill NaN values in 'Price' with the median
median_price = df_cleaned['Price'].median()
df_cleaned['Price'] = df_cleaned['Price'].fillna(median_price)

print("Price column after cleaning and imputation:")
print(df_cleaned['Price'].head())
print(df_cleaned['Price'].info())

Price column after cleaning and imputation:
0     9999
1     9990
2    11999
3    11999
4    11999
Name: Price, dtype: int64
<class 'pandas.core.series.Series'>
RangeIndex: 1370 entries, 0 to 1369
Series name: Price
Non-Null Count  Dtype
--------------  -----
1370 non-null   int64
dtypes: int64(1)
memory usage: 10.8 KB
None


In [4]:
import numpy as np

# 2. Clean 'Ram' column
def clean_ram(ram_str):
    if isinstance(ram_str, str):
        ram_str = ram_str.lower()
        if 'gb ram' in ram_str:
            val = ram_str.split('gb ram')[0].strip()
            if val.replace('.', '', 1).isdigit(): # Check if it's a number
                return float(val)
        elif 'mb ram' in ram_str:
            val = ram_str.split('mb ram')[0].strip()
            if val.replace('.', '', 1).isdigit(): # Check if it's a number
                return float(val) / 1024 # Convert MB to GB
        elif 'gb inbuilt' in ram_str or 'gb internal' in ram_str:
            # These seem to be misclassified inbuilt memory, treat as NaN for RAM
            return np.nan
        elif 'inbuilt' in ram_str or 'battery' in ram_str or 'helio' in ram_str:
            # Clearly not RAM values, convert to NaN
            return np.nan
        elif ram_str.replace('.', '', 1).isdigit(): # If it's just a number, assume GB
            return float(ram_str)
    return np.nan

df_cleaned['Ram_GB'] = df_cleaned['Ram'].apply(clean_ram)

print("Ram column after cleaning:")
print(df_cleaned[['Ram', 'Ram_GB']].head(10))
print(df_cleaned['Ram_GB'].info())
print(df_cleaned['Ram_GB'].value_counts(dropna=False).head())


Ram column after cleaning:
        Ram  Ram_GB
0  4 GB RAM     4.0
1  2 GB RAM     2.0
2  4 GB RAM     4.0
3  4 GB RAM     4.0
4  4 GB RAM     4.0
5  6 GB RAM     6.0
6  4 GB RAM     4.0
7  4 GB RAM     4.0
8  4 GB RAM     4.0
9  6 GB RAM     6.0
<class 'pandas.core.series.Series'>
RangeIndex: 1370 entries, 0 to 1369
Series name: Ram_GB
Non-Null Count  Dtype  
--------------  -----  
1350 non-null   float64
dtypes: float64(1)
memory usage: 10.8 KB
None
Ram_GB
8.0     528
4.0     253
12.0    246
6.0     214
16.0     41
Name: count, dtype: int64


In [5]:
import re

# 3. Clean 'Battery' column
def clean_battery(battery_str):
    if isinstance(battery_str, str):
        battery_str = battery_str.lower()
        # Look for 'mAh Battery' pattern
        match = re.search(r'(\d+)\s*mah battery', battery_str)
        if match:
            return float(match.group(1))
        # Also consider cases where it's just 'mAh'
        match = re.search(r'(\d+)\s*mah', battery_str)
        if match:
            return float(match.group(1))
    return np.nan

df_cleaned['Battery_mAh'] = df_cleaned['Battery'].apply(clean_battery)

print("Battery column after cleaning:")
print(df_cleaned[['Battery', 'Battery_mAh']].head(10))
print(df_cleaned['Battery_mAh'].info())
print(df_cleaned['Battery_mAh'].value_counts(dropna=False).head())


Battery column after cleaning:
             Battery  Battery_mAh
0  6000 mAh Battery        6000.0
1  4000 mAh Battery        4000.0
2  5000 mAh Battery        5000.0
3   6000 mAh Battery       6000.0
4  5000 mAh Battery        5000.0
5  5000 mAh Battery        5000.0
6  6000 mAh Battery        6000.0
7  5000 mAh Battery        5000.0
8  5000 mAh Battery        5000.0
9  6000 mAh Battery        6000.0
<class 'pandas.core.series.Series'>
RangeIndex: 1370 entries, 0 to 1369
Series name: Battery_mAh
Non-Null Count  Dtype  
--------------  -----  
1368 non-null   float64
dtypes: float64(1)
memory usage: 10.8 KB
None
Battery_mAh
5000.0    752
4500.0    117
6000.0    105
4000.0     44
4600.0     33
Name: count, dtype: int64


In [6]:
import re

# 4. Clean 'Display' column
def clean_display(display_str):
    if isinstance(display_str, str):
        display_str = display_str.lower()
        # Look for 'X inches' pattern where X is a number (can be float)
        match = re.search(r'(\d+\.?\d*)\s*inches', display_str)
        if match:
            return float(match.group(1))
    return np.nan

df_cleaned['Display_inches'] = df_cleaned['Display'].apply(clean_display)

print("Display column after cleaning:")
print(df_cleaned[['Display', 'Display_inches']].head(10))
print(df_cleaned['Display_inches'].info())
print(df_cleaned['Display_inches'].value_counts(dropna=False).head())

Display column after cleaning:
      Display  Display_inches
0  6.6 inches             6.6
1  6.4 inches             6.4
2  6.6 inches             6.6
3  6.4 inches             6.4
4  6.5 inches             6.5
5  6.5 inches             6.5
6  6.4 inches             6.4
7  6.5 inches             6.5
8  6.6 inches             6.6
9  6.6 inches             6.6
<class 'pandas.core.series.Series'>
RangeIndex: 1370 entries, 0 to 1369
Series name: Display_inches
Non-Null Count  Dtype  
--------------  -----  
1368 non-null   float64
dtypes: float64(1)
memory usage: 10.8 KB
None
Display_inches
6.70    158
6.67    151
6.78    142
6.50    122
6.60     94
Name: count, dtype: int64


In [7]:
import re
import numpy as np

# 5. Clean 'External_Memory' column
def clean_external_memory(mem_str):
    if isinstance(mem_str, str):
        mem_str = mem_str.lower()
        # Handle 'Memory Card Supported' without explicit size as NaN or 0, for now let's treat it as no specific size extracted
        if 'memory card supported' in mem_str and 'upto' not in mem_str:
            return np.nan

        # Look for 'upto X TB' pattern
        match_tb = re.search(r'upto\s*(\d+)\s*tb', mem_str)
        if match_tb:
            return float(match_tb.group(1)) * 1024 # Convert TB to GB

        # Look for 'upto X GB' pattern
        match_gb = re.search(r'upto\s*(\d+)\s*gb', mem_str)
        if match_gb:
            return float(match_gb.group(1))

        # Look for 'X GB' or 'X TB' without 'upto'
        match_gb_simple = re.search(r'(\d+)\s*gb', mem_str)
        if match_gb_simple:
            return float(match_gb_simple.group(1))

        match_tb_simple = re.search(r'(\d+)\s*tb', mem_str)
        if match_tb_simple:
            return float(match_tb_simple.group(1)) * 1024

    return np.nan

df_cleaned['External_Memory_GB'] = df_cleaned['External_Memory'].apply(clean_external_memory)

print("External Memory column after cleaning:")
print(df_cleaned[['External_Memory', 'External_Memory_GB']].head(10))
print(df_cleaned['External_Memory_GB'].info())
print(df_cleaned['External_Memory_GB'].value_counts(dropna=False).head())

External Memory column after cleaning:
                      External_Memory  External_Memory_GB
0    Memory Card Supported, upto 1 TB              1024.0
1  Memory Card Supported, upto 512 GB               512.0
2    Memory Card Supported, upto 1 TB              1024.0
3    Memory Card Supported, upto 1 TB              1024.0
4    Memory Card Supported, upto 1 TB              1024.0
5    Memory Card Supported, upto 1 TB              1024.0
6  Memory Card Supported, upto 512 GB               512.0
7               Memory Card Supported                 NaN
8    Memory Card Supported, upto 1 TB              1024.0
9    Memory Card Supported, upto 1 TB              1024.0
<class 'pandas.core.series.Series'>
RangeIndex: 1370 entries, 0 to 1369
Series name: External_Memory_GB
Non-Null Count  Dtype  
--------------  -----  
610 non-null    float64
dtypes: float64(1)
memory usage: 10.8 KB
None
External_Memory_GB
NaN       760
1024.0    394
512.0      87
256.0      75
2048.0     47
Name: count,

In [8]:
import re
import numpy as np

# 6. Clean 'Inbuilt_memory' column
def clean_inbuilt_memory(mem_str):
    if isinstance(mem_str, str):
        mem_str = mem_str.lower()
        # Look for 'X GB inbuilt' or 'X GB' pattern
        match_gb = re.search(r'(\d+)\s*gb(?: inbuilt)?', mem_str)
        if match_gb:
            return float(match_gb.group(1))

        # Look for 'X TB inbuilt' or 'X TB' pattern
        match_tb = re.search(r'(\d+)\s*tb(?: inbuilt)?', mem_str)
        if match_tb:
            return float(match_tb.group(1)) * 1024 # Convert TB to GB

    return np.nan

df_cleaned['Inbuilt_Memory_GB'] = df_cleaned['Inbuilt_memory'].apply(clean_inbuilt_memory)

print("Inbuilt Memory column after cleaning:")
print(df_cleaned[['Inbuilt_memory', 'Inbuilt_Memory_GB']].head(10))
print(df_cleaned['Inbuilt_Memory_GB'].info())
print(df_cleaned['Inbuilt_Memory_GB'].value_counts(dropna=False).head())

Inbuilt Memory column after cleaning:
    Inbuilt_memory  Inbuilt_Memory_GB
0   128 GB inbuilt              128.0
1    32 GB inbuilt               32.0
2    64 GB inbuilt               64.0
3    64 GB inbuilt               64.0
4    64 GB inbuilt               64.0
5   128 GB inbuilt              128.0
6    64 GB inbuilt               64.0
7    64 GB inbuilt               64.0
8    64 GB inbuilt               64.0
9   128 GB inbuilt              128.0
<class 'pandas.core.series.Series'>
RangeIndex: 1370 entries, 0 to 1369
Series name: Inbuilt_Memory_GB
Non-Null Count  Dtype  
--------------  -----  
1350 non-null   float64
dtypes: float64(1)
memory usage: 10.8 KB
None
Inbuilt_Memory_GB
128.0    644
256.0    405
64.0     184
512.0     59
32.0      48
Name: count, dtype: int64


In [9]:
import re
import numpy as np

# 7. Clean 'fast_charging' column
def clean_fast_charging(charging_str):
    if isinstance(charging_str, str):
        charging_str = charging_str.lower()
        # Look for 'XW Fast Charging' or 'XW Charging' pattern
        match = re.search(r'(\d+)\s*w(?: fast charging| charging)?', charging_str)
        if match:
            return float(match.group(1))
    return np.nan

df_cleaned['Fast_Charging_W'] = df_cleaned['fast_charging'].apply(clean_fast_charging)

print("Fast Charging column after cleaning:")
print(df_cleaned[['fast_charging', 'Fast_Charging_W']].head(10))
print(df_cleaned['Fast_Charging_W'].info())
print(df_cleaned['Fast_Charging_W'].value_counts(dropna=False).head())

Fast Charging column after cleaning:
        fast_charging  Fast_Charging_W
0   25W Fast Charging             25.0
1   15W Fast Charging             15.0
2   25W Fast Charging             25.0
3                 NaN              NaN
4   15W Fast Charging             15.0
5   15W Fast Charging             15.0
6   15W Fast Charging             15.0
7   15W Fast Charging             15.0
8   15W Fast Charging             15.0
9   15W Fast Charging             15.0
<class 'pandas.core.series.Series'>
RangeIndex: 1370 entries, 0 to 1369
Series name: Fast_Charging_W
Non-Null Count  Dtype  
--------------  -----  
1241 non-null   float64
dtypes: float64(1)
memory usage: 10.8 KB
None
Fast_Charging_W
18.0    157
33.0    144
NaN     129
67.0    101
25.0     97
Name: count, dtype: int64


In [10]:
import re
import numpy as np

# 8. Clean 'Processor' column
def clean_processor(processor_str):
    if isinstance(processor_str, str):
        processor_str = processor_str.lower()
        # Look for 'X GHz Processor' or 'X GHz' pattern
        match = re.search(r'(\d+\.?\d*)\s*ghz(?: processor)?', processor_str)
        if match:
            return float(match.group(1))
        # Also consider cases like 'Octa Core' which don't have a GHz speed
        if 'octa core' in processor_str or 'dual core' in processor_str or 'quad core' in processor_str:
            return np.nan # Or can be treated as a categorical feature later
    return np.nan

df_cleaned['Processor_GHz'] = df_cleaned['Processor'].apply(clean_processor)

print("Processor column after cleaning:")
print(df_cleaned[['Processor', 'Processor_GHz']].head(10))
print(df_cleaned['Processor_GHz'].info())
print(df_cleaned['Processor_GHz'].value_counts(dropna=False).head())

Processor column after cleaning:
              Processor  Processor_GHz
0   Octa Core Processor            NaN
1     1.8 GHz Processor            1.8
2       2 GHz Processor            2.0
3             Octa Core            NaN
4             Octa Core            NaN
5             Octa Core            NaN
6             Octa Core            NaN
7             Octa Core            NaN
8             Octa Core            NaN
9             Octa Core            NaN
<class 'pandas.core.series.Series'>
RangeIndex: 1370 entries, 0 to 1369
Series name: Processor_GHz
Non-Null Count  Dtype  
--------------  -----  
8 non-null      float64
dtypes: float64(1)
memory usage: 10.8 KB
None
Processor_GHz
NaN    1362
1.6       3
2.0       2
1.8       1
1.3       1
Name: count, dtype: int64


In [11]:
import re

# 9. Standardize 'company' column
def standardize_company_name(company_name):
    if isinstance(company_name, str):
        company_name = company_name.strip().lower()
        if company_name == 'iqoo':
            return 'iQOO'
        elif company_name == 'poco':
            return 'Poco'
        elif company_name == 'oneplus':
            return 'OnePlus'
        elif company_name == 'oppo':
            return 'Oppo'
        elif company_name == 'realme':
            return 'Realme'
        elif company_name == 'samsung':
            return 'Samsung'
        elif company_name == 'xiaomi':
            return 'Xiaomi'
        elif company_name == 'vivo':
            return 'Vivo'
        elif company_name == 'apple':
            return 'Apple'
        elif company_name == 'google':
            return 'Google'
        elif company_name == 'motorola':
            return 'Motorola'
        elif company_name == 'huawei':
            return 'Huawei'
        elif company_name == 'infinix':
            return 'Infinix'
        elif company_name == 'lg':
            return 'LG'
        elif company_name == 'micromax':
            return 'Micromax'
        elif company_name == 'nokia':
            return 'Nokia'
        elif company_name == 'sony':
            return 'Sony'
        elif company_name == 'tecno':
            return 'Tecno'
        elif company_name == 'asus':
            return 'Asus'
        elif company_name == 'htc':
            return 'HTC'
        elif company_name == 'honor':
            return 'Honor'
        # For other names, convert to Title Case as a default standardization
        return company_name.title()
    return company_name

df_cleaned['company'] = df_cleaned['company'].apply(standardize_company_name)

print("Company column after standardization:")
print(df_cleaned['company'].value_counts())
print(df_cleaned['company'].head())

Company column after standardization:
company
Vivo        186
Realme      186
Samsung     181
Motorola    127
Poco         94
Xiaomi       90
Honor        88
OnePlus      75
Oppo         65
Huawei       62
iQOO         58
Tcl          26
Google       23
Asus         21
Lava         19
Nothing      15
Itel         15
Lenovo       14
Tecno        13
LG            6
Gionee        5
Coolpad       1
Name: count, dtype: int64
0    Samsung
1    Samsung
2    Samsung
3    Samsung
4    Samsung
Name: company, dtype: object


In [12]:
import numpy as np

# 10. Handle missing values for remaining columns

# Impute 'Android_version' with mode
mode_android_version = df_cleaned['Android_version'].mode()[0]
df_cleaned['Android_version'] = df_cleaned['Android_version'].fillna(mode_android_version)

# Impute 'Screen_resolution' with mode
mode_screen_resolution = df_cleaned['Screen_resolution'].mode()[0]
df_cleaned['Screen_resolution'] = df_cleaned['Screen_resolution'].fillna(mode_screen_resolution)

# Impute 'Processor' (original categorical) with mode, as Processor_GHz had too many NaNs
mode_processor = df_cleaned['Processor'].mode()[0]
df_cleaned['Processor'] = df_cleaned['Processor'].fillna(mode_processor)

# Impute numerical cleaned columns with median
median_inbuilt_memory_gb = df_cleaned['Inbuilt_Memory_GB'].median()
df_cleaned['Inbuilt_Memory_GB'] = df_cleaned['Inbuilt_Memory_GB'].fillna(median_inbuilt_memory_gb)

median_fast_charging_w = df_cleaned['Fast_Charging_W'].median()
df_cleaned['Fast_Charging_W'] = df_cleaned['Fast_Charging_W'].fillna(median_fast_charging_w)

# The other numerical cleaned columns (Ram_GB, Battery_mAh, Display_inches, External_Memory_GB) were handled during their creation or already have very few NaNs.
# Let's check and impute them with median if any NaNs remain
median_ram_gb = df_cleaned['Ram_GB'].median()
df_cleaned['Ram_GB'] = df_cleaned['Ram_GB'].fillna(median_ram_gb)

median_battery_mah = df_cleaned['Battery_mAh'].median()
df_cleaned['Battery_mAh'] = df_cleaned['Battery_mAh'].fillna(median_battery_mah)

median_display_inches = df_cleaned['Display_inches'].median()
df_cleaned['Display_inches'] = df_cleaned['Display_inches'].fillna(median_display_inches)

median_external_memory_gb = df_cleaned['External_Memory_GB'].median()
df_cleaned['External_Memory_GB'] = df_cleaned['External_Memory_GB'].fillna(median_external_memory_gb)


print("Missing values after imputation:")
print(df_cleaned.isnull().sum()[df_cleaned.isnull().sum() > 0])
print("\nFirst 5 rows of the DataFrame after imputation:")
print(df_cleaned.head())

Missing values after imputation:
Inbuilt_memory      19
fast_charging       89
Processor_GHz     1362
dtype: int64

First 5 rows of the DataFrame after imputation:
   index                                  Name  Rating  Spec_score  \
0      0                 Samsung Galaxy F14 5G    4.65          68   
1      1                    Samsung Galaxy A11    4.20          63   
2      2                    Samsung Galaxy A13    4.30          75   
3      3                    Samsung Galaxy F23    4.10          73   
4      4  Samsung Galaxy A03s (4GB RAM + 64GB)    4.10          69   

                       No_of_sim       Ram            Battery     Display  \
0  Dual Sim, 3G, 4G, 5G, VoLTE,   4 GB RAM  6000 mAh Battery   6.6 inches   
1      Dual Sim, 3G, 4G, VoLTE,   2 GB RAM  4000 mAh Battery   6.4 inches   
2      Dual Sim, 3G, 4G, VoLTE,   4 GB RAM  5000 mAh Battery   6.6 inches   
3      Dual Sim, 3G, 4G, VoLTE,   4 GB RAM   6000 mAh Battery  6.4 inches   
4      Dual Sim, 3G, 4G, VoLTE

In [13]:
print("Data types of all columns in df_cleaned:")
df_cleaned.info()

Data types of all columns in df_cleaned:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1370 entries, 0 to 1369
Data columns (total 25 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   index               1370 non-null   int64  
 1   Name                1370 non-null   object 
 2   Rating              1370 non-null   float64
 3   Spec_score          1370 non-null   int64  
 4   No_of_sim           1370 non-null   object 
 5   Ram                 1370 non-null   object 
 6   Battery             1370 non-null   object 
 7   Display             1370 non-null   object 
 8   Camera              1370 non-null   object 
 9   External_Memory     1370 non-null   object 
 10  Android_version     1370 non-null   object 
 11  Price               1370 non-null   int64  
 12  company             1370 non-null   object 
 13  Inbuilt_memory      1351 non-null   object 
 14  fast_charging       1281 non-null   object 
 15  Screen_resolut

In [14]:
categorical_cols_for_analysis = [
    'Name', 'No_of_sim', 'Android_version', 'company',
    'Screen_resolution', 'Processor', 'Processor_name', 'Camera'
]
numerical_cols_for_analysis = [
    'Rating', 'Spec_score', 'Price', 'Ram_GB', 'Battery_mAh',
    'Display_inches', 'External_Memory_GB', 'Inbuilt_Memory_GB',
    'Fast_Charging_W', 'Processor_GHz'
]

print("\nUnique values and counts for specified categorical columns:")
for col in categorical_cols_for_analysis:
    print(f"\nColumn: {col}")
    # Limit output for columns with many unique values like 'Name', 'Camera', 'Screen_resolution'
    if df_cleaned[col].nunique() > 50:
        print(f"Total unique values: {df_cleaned[col].nunique()}")
        print(df_cleaned[col].value_counts(dropna=False).head(10)) # Print top 10
    else:
        print(df_cleaned[col].value_counts(dropna=False))

print("\nDescriptive statistics for specified numerical columns:")
print(df_cleaned[numerical_cols_for_analysis].describe())

print("\nUnique values in 'Name' column (top 10 for brevity):")
print(df_cleaned['Name'].value_counts().head(10))

print("\nUnique values in 'company' column:")
print(df_cleaned['company'].value_counts())


Unique values and counts for specified categorical columns:

Column: Name
Total unique values: 1334
Name
Realme Narzo N55 (6GB RAM + 128GB)          3
Vivo S19 Pro                                2
Samsung Galaxy S21 FE (Snapdragon)          2
Vivo V25 Pro (12GB RAM + 256GB)             2
Vivo V30 5G                                 2
Xiaomi Redmi Note 11 (6GB RAM + 128GB)      2
Realme Narzo N53                            2
Realme 12 Pro Plus 5G (12GB RAM + 256GB)    2
Realme 12 Pro Plus 5G (8GB RAM + 256GB)     2
Vivo Y56 5G (4GB RAM + 128GB)               2
Name: count, dtype: int64

Column: No_of_sim
No_of_sim
Dual Sim, 3G, 4G, 5G, VoLTE,             818
Dual Sim, 3G, 4G, VoLTE,                 419
Dual Sim, 3G, 4G, 5G, VoLTE, Vo5G,        91
Single Sim, 3G, 4G, 5G, VoLTE,            20
Dual Sim, 3G, 4G,                         13
Single Sim, 3G, 4G, VoLTE,                 6
No Sim Supported,                          1
Single Sim, 3G, 4G, 5G, VoLTE, Vo5G,       1
Dual Sim, 3G, VoLTE

In [15]:
columns_for_synthetic_data = [
    'Name', 'Rating', 'Spec_score', 'No_of_sim', 'Android_version',
    'Price', 'company', 'Camera', 'Screen_resolution', 'Processor_name',
    'Ram_GB', 'Battery_mAh', 'Display_inches', 'External_Memory_GB',
    'Inbuilt_Memory_GB', 'Fast_Charging_W', 'Processor_GHz'
]

print("Columns selected for synthetic data generation:")
print(columns_for_synthetic_data)

Columns selected for synthetic data generation:
['Name', 'Rating', 'Spec_score', 'No_of_sim', 'Android_version', 'Price', 'company', 'Camera', 'Screen_resolution', 'Processor_name', 'Ram_GB', 'Battery_mAh', 'Display_inches', 'External_Memory_GB', 'Inbuilt_Memory_GB', 'Fast_Charging_W', 'Processor_GHz']


In [16]:
df_synthetic = df_cleaned[columns_for_synthetic_data].sample(n=100000, replace=True, random_state=42)

print("First 5 rows of df_synthetic:")
print(df_synthetic.head())

print("\nConcise summary of df_synthetic:")
df_synthetic.info()

First 5 rows of df_synthetic:
                                    Name  Rating  Spec_score  \
1126     OnePlus 11 Jupiter Rock Edition    4.15          90   
860                     Realme Narzo N53    4.75          70   
1294                            Honor X5    4.20          60   
1130  Xiaomi Redmi 13C (6GB RAM + 128GB)    4.60          75   
1095                       OnePlus Ace 2    4.50          87   

                          No_of_sim Android_version  Price  company  \
1126  Dual Sim, 3G, 4G, 5G, VoLTE,               13  64990  OnePlus   
860       Dual Sim, 3G, 4G, VoLTE,               13   7820   Realme   
1294      Dual Sim, 3G, 4G, VoLTE,               12   8990    Honor   
1130      Dual Sim, 3G, 4G, VoLTE,               13   8499   Xiaomi   
1095  Dual Sim, 3G, 4G, 5G, VoLTE,               13  34999  OnePlus   

                                                 Camera  \
1126  50 MP + 48 MP + 32 MP Triple Rear &amp; 16 MP ...   
860   50 MP + Depth Sensor Dual Rear &am

In [17]:
df_synthetic.to_csv('generated_mobile_data_100k.csv', index=False)

print("Synthetic data saved to generated_mobile_data_100k.csv")

Synthetic data saved to generated_mobile_data_100k.csv


In [7]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# กำหนดจำนวนข้อมูลที่ต้องการ
N = 100000

# 1. จำลองข้อมูลพื้นฐาน
companies = ['OnePlus', 'Realme', 'Honor', 'Xiaomi', 'Motorola', 'Vivo', 'Samsung', 'Oppo']
processors = ['Snapdragon 8 Gen2', 'Unisoc T612', 'Helio G25', 'Helio G85', 'Snapdragon 8+ Gen1', 'Exynos 2200']
sim_types = ["Dual Sim, 3G, 4G, VoLTE", "Dual Sim, 3G, 4G, 5G, VoLTE"]
screens = ['720 x 1600 px', '1080 x 2400 px', '1240 x 2772 px', '1440 x 3216 px']

random_companies = np.random.choice(companies, N)
model_codes = [''.join(np.random.choice(list('ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789'), 4)) for _ in range(N)]
names = [f"{comp} Model {code}" for comp, code in zip(random_companies, model_codes)]

def random_dates(start_date, end_date, n):
    time_between_dates = end_date - start_date
    days_between_dates = time_between_dates.days
    random_number_of_days = np.random.randint(0, days_between_dates, n)
    return [start_date + timedelta(days=int(days)) for days in random_number_of_days]

start_dt = datetime(2021, 1, 1)
end_dt = datetime(2024, 1, 1)
release_dates = random_dates(start_dt, end_dt, N)

# 2. สร้าง DataFrame โดยเพิ่มคอลัมน์ดั้งเดิมให้ครบ 100%
df = pd.DataFrame({
    'Name': names,
    'Rating': np.round(np.random.uniform(3.5, 5.0, N), 2),
    'Spec_score': np.random.randint(50, 99, N),
    'No_of_sim': np.random.choice(sim_types, N),
    'Android_version': np.random.choice([11, 12, 13, 14], N),
    'Price': np.random.randint(4000, 50000, N),
    'company': random_companies,
    'Processor_name': np.random.choice(processors, N),
    'Ram_GB': np.random.choice([4.0, 6.0, 8.0, 12.0, 16.0], N),
    'Battery_mAh': np.random.choice([4000.0, 4500.0, 5000.0, 6000.0], N),
    'Display_inches': np.round(np.random.uniform(6.1, 6.9, N), 2),
    'Inbuilt_Memory_GB': np.random.choice([64.0, 128.0, 256.0, 512.0], N),
    
    # --- เพิ่มคอลัมน์ดั้งเดิมที่คุณใช้ในกราฟเก่าเพื่อป้องกัน KeyError ---
    'Camera': np.random.choice([12, 16, 32, 48, 50, 64, 108], N),  # สุ่มความละเอียดกล้อง (MP)
    'Screen_resolution': np.random.choice(screens, N),
    'External_Memory_GB': np.random.choice([0.0, 256.0, 512.0, 1024.0, 2048.0], N),
    'Fast_Charging_W': np.random.choice([18.0, 33.0, 44.0, 65.0, 100.0, 120.0, 125.0], N),
    'Processor_GHz': np.round(np.random.uniform(1.8, 3.2, N), 2),
    
    # --- คอลัมน์ Big Data (ใหม่) ---
    'Release_Date': release_dates,
    'Monthly_Sales_Volume': np.random.randint(100, 50000, N),
})

df['Ecom_Page_Views'] = df['Monthly_Sales_Volume'] * np.random.randint(20, 100, N)
df['Add_to_Cart_Rate'] = np.round(np.random.uniform(0.01, 0.15, N), 4)
df['Release_Date'] = pd.to_datetime(df['Release_Date']).dt.strftime('%Y-%m-%d')

# บันทึกไฟล์ทับของเดิม
output_filename = "simulated_smartphone_bigdata_100k.csv"
df.to_csv(output_filename, index=False)

print("✅ เจ็นข้อมูล 100k ใหม่และเพิ่มคอลัมน์ Camera ให้เรียบร้อยแล้ว!")

✅ เจ็นข้อมูล 100k ใหม่และเพิ่มคอลัมน์ Camera ให้เรียบร้อยแล้ว!
